# 🧠 Knowledge Distillation for FLAN-T5 Compression

This notebook explores transferring the clinical reasoning capability of large teacher models (**FLAN-T5-XL / FLAN-T5-large**) into smaller, latency-efficient student models (**FLAN-T5-base**).

### Key Architectural Learnings:
- **Offline Logits Precomputation Bottleneck**: In Seq2Seq models with 32,128 vocabulary tokens and sequences of 512 tokens, precomputing and saving full logits requires ~65 MB per sample ($512 \times 32128 \times 4$ bytes). For 1,000 samples, this exceeds 65 GB in RAM, resulting in `MemoryError` when converting to Python lists.
- **Online Knowledge Distillation Solution**: We execute online distillation where teacher logits are computed on-the-fly per batch under `torch.no_grad()`, computing the KL divergence loss directly without persisting massive logits matrices.

In [ ]:
import sys
from pathlib import Path
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

REPO_ROOT = Path("..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.distillation.loss import DistillationLoss
from src.distillation.distillation_trainer import Seq2SeqDistillationTrainer
from src.data.dataset import load_clinical_splits

print("CUDA Available for Distillation:", torch.cuda.is_available())

## 1. Distillation Loss Formulation
$$\mathcal{L}_{total} = \alpha \cdot T^2 \cdot \mathcal{D}_{KL}(P_{student}(T) \parallel P_{teacher}(T)) + (1 - \alpha) \cdot \mathcal{L}_{CE}(student, y)$$

In [ ]:
# Instantiate distillation loss
distill_loss = DistillationLoss(temperature=2.0, alpha=0.5, ignore_index=-100)
print("Distillation loss initialized with Temperature=2.0, Alpha=0.5")

## 2. Load Teacher and Student Models

In [ ]:
teacher_id = "google/flan-t5-large"
student_id = "google/flan-t5-base"

print(f"Teacher: {teacher_id}")
print(f"Student: {student_id}")

tokenizer = AutoTokenizer.from_pretrained(student_id)
# To load models for training:
# teacher_model = AutoModelForSeq2SeqLM.from_pretrained(teacher_id)
# student_model = AutoModelForSeq2SeqLM.from_pretrained(student_id)

## 3. Online Distillation Training Pipeline

In [ ]:
# Load data splits
dataset_dict = load_clinical_splits(
    train_path="../data/processed/train.csv",
    val_path="../data/processed/val.csv",
    test_path="../data/processed/test.csv",
    tokenizer=tokenizer,
    max_input_length=512,
    max_target_length=128
)
print("Train samples:", len(dataset_dict['train']))
print("Val samples:", len(dataset_dict['validation']))

## 4. Run Distillation via CLI

To run distillation with automatic device management, execute:
```bash
python scripts/distill.py --config configs/distill_large_to_base.yaml
```